# 01 - Data Understanding & Ingestion
## NASA C-MAPSS Turbofan Engine Degradation Simulation Dataset (FD001)

### Project Context
- **Project Title**: Machine Learning-Based Remaining Useful Life Estimation for Turbofan Engines
- **Task Type**: Supervised Regression
- **Target Variable**: Remaining Useful Life (RUL) in operational flight cycles
- **Dataset Subset**: FD001 (Sea level conditions, High-Pressure Compressor failure mode)

### Notebook Objectives
1. Ingest raw space-separated text files via modular data loader (`src/data_loader.py`).
2. Verify dimensional shapes and unique engine counts across training, test, and ground truth sets.
3. Inspect schema types, first/last rows, and memory footprints.
4. Compute descriptive statistics across all 26 telemetry channels.
5. Audit data integrity: check for missing values, duplicate rows, and duplicate cycle indices.
6. Verify no data leakage or premature transformations occur at this stage.

*Note: In accordance with project requirements, no ML models are trained, and raw values remain unmodified.*

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from src.data_loader import load_cmapss_data

# Set display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

print(f"Project root identified at: {project_root.resolve()}")

---
## 1. Load Raw Dataset via `src/data_loader.py`

We invoke the centralized data loader function `load_cmapss_data()`. This function parses the space-delimited text files, trims any accidental spacing-induced columns, assigns standardized column headers, and leaves all numerical values intact.

In [ ]:
train_df, test_df, rul_df = load_cmapss_data(data_dir=project_root / "data" / "raw")
print("Dataset successfully ingested via src.data_loader!")

---
## 2. Dataset Dimensions & Unique Engine Counts

In [ ]:
print("=" * 60)
print("                   DATASET DIMENSIONS")
print("=" * 60)
print(f"Training Trajectories (train_FD001.txt) : {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Test Trajectories (test_FD001.txt)       : {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")
print(f"Ground Truth RUL (RUL_FD001.txt)        : {rul_df.shape[0]:,} rows x {rul_df.shape[1]} column")

print("\n" + "=" * 60)
print("                 UNIQUE ENGINE UNITS")
print("=" * 60)
print(f"Unique engines in Training Set : {train_df['unit_number'].nunique()} (Engine IDs: {train_df['unit_number'].min()} to {train_df['unit_number'].max()})")
print(f"Unique engines in Test Set     : {test_df['unit_number'].nunique()} (Engine IDs: {test_df['unit_number'].min()} to {test_df['unit_number'].max()})")
print(f"Target evaluations in RUL Set  : {len(rul_df)} ground truth values")

---
## 3. Data Preview: First and Last 5 Rows

Examining snapshots of the beginning and end of the dataset allows us to verify row structure and telemetry formatting.

In [ ]:
print("--- FIRST 5 ROWS (Training Set) ---")
train_df.head()

In [ ]:
print("--- LAST 5 ROWS (Training Set) ---")
train_df.tail()

---
## 4. Column Data Types & Memory Usage

In [ ]:
print("--- DATA TYPES & NON-NULL SUMMARY ---")
train_df.info()

memory_mb = train_df.memory_usage().sum() / (1024 ** 2)
print(f"\nTotal DataFrame Memory Footprint: {memory_mb:.2f} MB")

---
## 5. Descriptive Statistics

We inspect the five-number summary (mean, std, min, 25%, 50%, 75%, max) across operational settings and 21 sensors.

In [ ]:
train_df.describe().T

---
## 6. Operating Cycles per Engine (Lifetime Analysis)

In the training set, each engine trajectory represents a continuous run-to-failure cycle. The final recorded cycle for each engine is its operational lifespan.

In [ ]:
cycles_per_unit = train_df.groupby('unit_number')['time_cycles'].max()

print("=" * 60)
print("             ENGINE LIFETIME SUMMARY (TRAINING SET)")
print("=" * 60)
print(f"Shortest Engine Lifetime : {cycles_per_unit.min()} cycles (Engine #{cycles_per_unit.idxmin()})")
print(f"Longest Engine Lifetime  : {cycles_per_unit.max()} cycles (Engine #{cycles_per_unit.idxmax()})")
print(f"Mean Engine Lifetime     : {cycles_per_unit.mean():.2f} cycles")
print(f"Median Engine Lifetime   : {cycles_per_unit.median():.1f} cycles")
print(f"Standard Deviation       : {cycles_per_unit.std():.2f} cycles")
print("-" * 60)
print("Sample of First 10 Engine Lifespans:")
for unit, max_c in cycles_per_unit.head(10).items():
    print(f"  Engine {unit:3d}: {max_c} operational cycles")

---
## 7. Data Quality & Integrity Audits

Checking for missing values (`NaN`), duplicate rows, and index collisions between `unit_number` and `time_cycles`.

In [ ]:
print("=" * 60)
print("                 DATA INTEGRITY CHECKS")
print("=" * 60)

# Check for nulls
null_train = train_df.isna().sum().sum()
null_test = test_df.isna().sum().sum()
null_rul = rul_df.isna().sum().sum()
print(f"Missing values in Training set : {null_train}")
print(f"Missing values in Test set     : {null_test}")
print(f"Missing values in RUL set      : {null_rul}")

# Check for duplicate rows
dup_train = train_df.duplicated().sum()
dup_test = test_df.duplicated().sum()
dup_rul = rul_df.duplicated().sum()
print(f"Duplicate records in Training  : {dup_train}")
print(f"Duplicate records in Test      : {dup_test}")
print(f"Duplicate records in RUL       : {dup_rul}")

# Check for duplicate (unit_number, time_cycles) pairs
pair_dups = train_df.duplicated(subset=['unit_number', 'time_cycles']).sum()
print(f"Duplicate (unit, cycle) pairs  : {pair_dups}")

assert null_train == 0, "Error: Unexpected missing values detected in training set!"
assert dup_train == 0, "Error: Duplicate rows found in training set!"
assert pair_dups == 0, "Error: Non-unique engine cycle snapshots detected!"
print("\nAll integrity assertions passed successfully!")

---
## 8. Summary of Findings (Data Understanding)

| Property | Value / Observation | ML Implication |
| :--- | :--- | :--- |
| **Training Records** | 20,631 cycles across 100 engines | Sufficient volume for regression modeling |
| **Test Records** | 13,096 cycles across 100 engines | Test data must remain unseen during feature engineering |
| **Engine Lifespans** | Min: 128 cycles, Max: 362 cycles, Mean: 206.31 | Variable engine wear rates, requiring normalized or sequence-aware features |
| **Missing Values** | Exactly 0 across all files | No imputation required |
| **Duplicates** | Exactly 0 duplicate records | Clean telemetry snapshots |
| **Data Types** | Integer (IDs, cycles), Float (Settings, Sensors) | Numeric representations ready for numerical processing |

*Next Step: Proceed to `02_eda.ipynb` to perform comprehensive exploratory data analysis, evaluate sensor variances, and observe degradation signatures.*